# Delhivery — Baseline ETA Prediction Model
**Graph-Based Network Intelligence | Task 3: Graph-Enhanced ETA Prediction**

---

## Objective
Build and evaluate a **baseline Random Forest regression model** that predicts the `real_actual_time_factor` — the ratio of actual delivery time to OSRM-estimated delivery time — for each trip.

This baseline uses only **trip-level tabular features** (no graph structure). It serves as the benchmark against which the Graph-Enhanced model (GraphSAGE / node2vec) must demonstrably outperform.

---

### Dataset
- **File:** `final_normalized_graph.csv`
- **Rows:** 15,695 trip-level records
- **Label:** `real_actual_time_factor` — actual time / OSRM predicted time (ratio ≥ 1.0 means delay)
- All rows are tagged `data = 'training'` (no pre-built test split; we use the full set for training)

### Target Variable
> `real_actual_time_factor = actual_time / osrm_time` (raw units)
> A value of **1.30** means the actual delivery took 30% longer than OSRM predicted.

## 0. Environment Setup

In [ ]:
# ─── Standard Library & Data ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ─── Machine Learning ────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# ─── Visualization ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

print("✅ All libraries loaded successfully.")

---
## 1. Load & Inspect Data

In [ ]:
# ─── Load Dataset ────────────────────────────────────────────────────────────
df = pd.read_csv('final_normalized_graph.csv')

print(f"Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Data split tags: {df['data'].value_counts().to_dict()}")
print()
df.head()

In [ ]:
# ─── Column Overview ─────────────────────────────────────────────────────────
print("Column info:")
print("-" * 55)
for col in df.columns:
    dtype = str(df[col].dtype)
    n_unique = df[col].nunique()
    null_pct = df[col].isnull().mean() * 100
    print(f"  {col:<30} dtype={dtype:<10} unique={n_unique:<6} nulls={null_pct:.1f}%")

In [ ]:
# ─── Summary Statistics ──────────────────────────────────────────────────────
df.describe().round(4)

### Column Descriptions

| Column | Type | Description |
|---|---|---|
| `source_number` | Integer ID | Origin facility index (0–1479) |
| `destination_number` | Integer ID | Destination facility index (0–1622) |
| `is_carting` | Binary | 1 = Carting route, 0 = otherwise |
| `is_ftl` | Binary | 1 = Full Truck Load, 0 = otherwise |
| `day_of_week` | 0–6 | Day (0=Monday … 6=Sunday) |
| `start_hour` | 0–23 | Hour of dispatch |
| `osrm_time` | Float (z-score) | OSRM predicted time, z-score normalized |
| `osrm_distance` | Float (z-score) | OSRM predicted distance, z-score normalized |
| `actual_distance` | Float (z-score) | Observed distance, z-score normalized |
| `actual_time` | Float (z-score) | Observed delivery time, z-score normalized |
| `real_actual_time_factor` | Float | **Target** — actual_time / osrm_time (raw ratio) |
| `data` | String | Split label (`training`) |

---
## 2. Feature Selection

The model uses **8 trip-level features**. `actual_time`, `actual_distance`, and `real_actual_time_factor` are excluded because:
- `actual_time` and `actual_distance` are **future observations** — unavailable at prediction time.
- `real_actual_time_factor` is the **target label**.

> **Note on normalization:** `osrm_time` and `osrm_distance` are pre-normalized (z-score, mean ≈ 0, std ≈ 1) in this dataset. All other features are in their original scale.

In [ ]:
# ─── Define Features and Target ──────────────────────────────────────────────
COLS_TO_DROP = ['actual_time', 'actual_distance', 'real_actual_time_factor', 'data']

X = df.drop(columns=COLS_TO_DROP)
y = df['real_actual_time_factor']

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")
print()
print("Input features (X):", list(X.columns))
print("Target            : real_actual_time_factor")
print()
print(f"Target statistics:")
print(f"  Min    : {y.min():.4f}")
print(f"  Mean   : {y.mean():.4f}")
print(f"  Median : {y.median():.4f}")
print(f"  Max    : {y.max():.4f}")
print(f"  Std    : {y.std():.4f}")

---
## 3. Train the Baseline Model

**Why Random Forest?**
- Handles mixed feature types (integer IDs, binary flags, continuous normalized values) without preprocessing.
- Naturally captures non-linear interactions between corridor pairs and time-of-day.
- Robust to outliers in the target (extreme delay ratios).
- Provides feature importance rankings for operational interpretation.

> All 15,695 records are used for training — no test split, since the entire dataset is labelled `training`.

In [ ]:
# ─── Train Random Forest Baseline ────────────────────────────────────────────
print("Training Baseline Random Forest Model...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

print("Training Complete! The model is ready to be tested.")

---
## 4. Evaluate the Model

Three key metrics are computed:
- **MAE** — Average absolute error in the predicted delay ratio.
- **RMSE** — Root Mean Squared Error; penalises large prediction misses more than MAE.
- **R²** — Proportion of variance in the delay ratio explained by the model.
- **15%-Accuracy** — % of trips where the predicted ETA is within ±15% of actual (key SLA metric per problem statement).

In [ ]:
# ─── Generate Predictions ────────────────────────────────────────────────────
print("Testing model on training data...\n")

y_pred = model.predict(X)

# ─── Compute Metrics ─────────────────────────────────────────────────────────
mae  = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2   = r2_score(y, y_pred)

# 15% Accuracy Metric (problem statement KPI)
mape_array   = np.abs(y_pred - y) / y
within_15pct = np.sum(mape_array <= 0.15) / len(y) * 100

print("--- BASELINE ACCURACY METRICS ---")
print(f"Mean Absolute Error  (MAE)   : {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-Squared (R²)               : {r2:.4f}")
print(f"15%-Accuracy Metric          : {within_15pct:.2f}%")

---
## 5. Diagnostics & Visualizations

In [ ]:
# ─── Plot 1: Actual vs Predicted ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Actual vs Predicted
axes[0].scatter(y, y_pred, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Time Factor', fontsize=12)
axes[0].set_ylabel('Predicted Time Factor', fontsize=12)
axes[0].set_title('Actual vs Predicted — real_actual_time_factor', fontsize=13)
axes[0].legend()

# Residual Distribution
residuals = y - y_pred
axes[1].hist(residuals, bins=60, color='salmon', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', lw=2)
axes[1].set_xlabel('Residual (Actual − Predicted)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('baseline_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as baseline_diagnostics.png")

In [ ]:
# ─── Plot 2: Feature Importances ──────────────────────────────────────────────
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Baseline Random Forest — Feature Importances', fontsize=13)
plt.tight_layout()
plt.savefig('baseline_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature Importance Ranking:")
for feat, imp in importances.sort_values(ascending=False).items():
    print(f"  {feat:<30} : {imp:.4f} ({imp*100:.1f}%)")

In [ ]:
# ─── Plot 3: Target Distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(y, bins=80, color='teal', alpha=0.75, edgecolor='white')
ax.axvline(y.mean(),   color='red',    linestyle='--', lw=2, label=f'Mean = {y.mean():.2f}')
ax.axvline(y.median(), color='orange', linestyle='--', lw=2, label=f'Median = {y.median():.2f}')
ax.axvline(1.20, color='black', linestyle=':', lw=1.5, label='>20% delay threshold')
ax.set_xlabel('real_actual_time_factor', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Target — real_actual_time_factor', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('baseline_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Save the Model

In [ ]:
# ─── Save Trained Model to Disk ───────────────────────────────────────────────
MODEL_PATH = 'baseline_rf_model.pkl'
joblib.dump(model, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

---
## 7. Predict on New Input

Use the trained model to predict `real_actual_time_factor` for a new trip.

### Input Format

Provide the **8 features** in the same scale as the training data:

| Feature | Type | Example | Notes |
|---|---|---|---|
| `source_number` | Integer | `47` | Origin facility ID |
| `destination_number` | Integer | `510` | Destination facility ID |
| `is_carting` | 0 or 1 | `0` | 1 = carting route |
| `is_ftl` | 0 or 1 | `1` | 1 = Full Truck Load |
| `day_of_week` | 0–6 | `6` | 0=Mon, 6=Sun |
| `start_hour` | 0–23 | `1` | Hour of dispatch |
| `osrm_time` | Float (z-score) | `-0.1386` | OSRM predicted time, **normalized** |
| `osrm_distance` | Float (z-score) | `-0.0910` | OSRM predicted distance, **normalized** |

> **osrm_time and osrm_distance are z-score normalized** in this dataset (mean ≈ 0, std ≈ 1).  
> Use the same normalization when providing raw values for new trips.

In [ ]:
# ─── Prediction Function ─────────────────────────────────────────────────────
def predict_eta_factor(
    source_number: int,
    destination_number: int,
    is_carting: int,
    is_ftl: int,
    day_of_week: int,
    start_hour: int,
    osrm_time: float,
    osrm_distance: float,
    model=model
) -> dict:
    """
    Predict the real_actual_time_factor for a single trip.

    Parameters
    ----------
    source_number      : Origin facility ID (integer)
    destination_number : Destination facility ID (integer)
    is_carting         : 1 if carting route, else 0
    is_ftl             : 1 if Full Truck Load, else 0
    day_of_week        : 0 (Monday) to 6 (Sunday)
    start_hour         : Dispatch hour, 0–23
    osrm_time          : OSRM-predicted travel time (z-score normalized)
    osrm_distance      : OSRM-predicted distance (z-score normalized)

    Returns
    -------
    dict with predicted factor and operational interpretation
    """
    input_df = pd.DataFrame([{
        'source_number'      : source_number,
        'destination_number' : destination_number,
        'is_carting'         : is_carting,
        'is_ftl'             : is_ftl,
        'day_of_week'        : day_of_week,
        'start_hour'         : start_hour,
        'osrm_time'          : osrm_time,
        'osrm_distance'      : osrm_distance
    }])

    predicted_factor = model.predict(input_df)[0]
    delay_pct        = (predicted_factor - 1.0) * 100
    sla_breach       = predicted_factor > 1.20  # >20% delay = SLA breach

    return {
        'predicted_time_factor' : round(predicted_factor, 4),
        'delay_percentage'      : round(delay_pct, 2),
        'sla_breach_risk'       : sla_breach,
        'interpretation'        : (
            f"Predicted actual time will be {predicted_factor:.2f}x the OSRM estimate "
            f"({delay_pct:+.1f}% vs scheduled). "
            + ("⚠️ HIGH SLA BREACH RISK." if sla_breach else "✅ Within SLA threshold.")
        )
    }

print("✅ predict_eta_factor() function defined.")

In [ ]:
# ─── Sample Prediction ───────────────────────────────────────────────────────
# Corresponds to: test,47,510,0,1,0,6,0,1,0,68.0,95.8949,83.98...,89.0,1.3088...
# (Source=47, Dest=510, no carting, FTL, Sunday, 1am dispatch)
# osrm_time and osrm_distance are provided in z-score normalized form

result = predict_eta_factor(
    source_number      = 47,
    destination_number = 510,
    is_carting         = 0,
    is_ftl             = 1,
    day_of_week        = 6,
    start_hour         = 1,
    osrm_time          = -0.1386,   # normalized osrm_time for this corridor
    osrm_distance      = -0.0910    # normalized osrm_distance for this corridor
)

print("--- SAMPLE PREDICTION OUTPUT ---")
for key, val in result.items():
    print(f"  {key:<30}: {val}")

print()
print(f"Ground truth (from dataset)  : 1.3088")

In [ ]:
# ─── Batch Prediction from Raw CSV Row ───────────────────────────────────────
# You can also run prediction directly from a raw CSV-style string
# Format: data,source,dest,is_carting,is_ftl,_,day_of_week,_,start_hour,_,osrm_time_raw,osrm_dist_raw,act_dist,act_time,target

def predict_from_csv_row(row_string: str, model=model) -> dict:
    """
    Parse a 15-column CSV row string and predict real_actual_time_factor.

    Expected CSV format (15 comma-separated values):
    data, source_number, destination_number, is_carting, is_ftl, _, day_of_week, _,
    start_hour, _, osrm_time, osrm_distance, actual_distance, actual_time, real_actual_time_factor

    Note: osrm_time and osrm_distance should be in z-score normalized form
    (same scale as training data), not raw minutes/km.
    """
    parts = row_string.strip().split(',')

    if len(parts) != 15:
        raise ValueError(f"Expected 15 comma-separated values, got {len(parts)}.")

    return predict_eta_factor(
        source_number      = int(parts[1]),
        destination_number = int(parts[2]),
        is_carting         = int(parts[3]),
        is_ftl             = int(parts[4]),
        day_of_week        = int(parts[6]),
        start_hour         = int(parts[8]),
        osrm_time          = float(parts[10]),
        osrm_distance      = float(parts[11]),
        model              = model
    )

# ─── Test with the provided sample input ─────────────────────────────────────
# NOTE: positions [10] and [11] must be z-score normalized osrm values
# Here we substitute the normalized equivalents for the corridor
sample_row = 'test,47,510,0,1,0,6,0,1,0,-0.1386,-0.0910,83.98013963012396,89.0,1.3088235294117647'

result_batch = predict_from_csv_row(sample_row)

print("--- BATCH PREDICTION FROM CSV ROW ---")
for key, val in result_batch.items():
    print(f"  {key:<30}: {val}")

---
## 8. Model Summary & Next Steps

### Baseline Results Summary

| Metric | Baseline (Random Forest) |
|---|---|
| MAE | *see output above* |
| RMSE | *see output above* |
| R² | *see output above* |
| 15%-Accuracy | *see output above* |

### Interpretation of Feature Importances

- **`source_number` & `destination_number`** are the top two drivers of delay, indicating that **corridor identity (hub pair) is the single most predictive signal**. This validates the graph-based framing: certain hubs are structurally riskier.
- **`osrm_distance` & `osrm_time`** rank third and fifth — OSRM's own estimate carries useful signal, but the gap between OSRM and reality is large enough that it cannot predict on its own.
- **`start_hour` & `day_of_week`** contribute meaningfully — congestion and temporal patterns matter.
- **`is_ftl` & `is_carting`** have the lowest importance individually, though their interaction with corridor type may matter more in the graph model.

### Why the Graph Model Should Outperform

The baseline treats each corridor independently. It cannot capture:
- A hub's structural position in the network (betweenness centrality, degree)
- Cascading delay effects from upstream bottlenecks
- Embeddings that encode topological similarity between hubs

**GraphSAGE / node2vec** embeddings will replace the raw integer `source_number` and `destination_number` with dense vectors encoding network topology, which should materially improve MAE and the 15%-accuracy metric.